# LeetCode 1908 — Game of Nim bằng Python

Link đề: https://leetcode.com/problems/game-of-nim/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Bit Manipulation, Brainteaser, Game Theory

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def nimGame(self, piles: List[int]) -> bool:
        ...
```

Lộ trình: **minimax đệ quy không nhớ** (vét cạn cây trò chơi) → **minimax có ghi nhớ trên trạng thái chuẩn hóa** → **DP từ dưới lên trên mã hóa cơ số 8** → **định lý Bouton / Sprague–Grundy: Alice thắng ⇔ XOR các đống ≠ 0** `O(n)` (trả lời follow-up).

## 1. Mục tiêu học

1. Hiểu khái niệm **trạng thái thắng / thua** (W/L) trong trò chơi hai người chơi tối ưu: một trạng thái là **thắng** nếu tồn tại **một** nước đi tới trạng thái **thua**; là **thua** nếu **mọi** nước đi đều tới trạng thái thắng.
2. Cài đặt minimax bằng đệ quy + `lru_cache`, **chuẩn hóa trạng thái** (sắp xếp) để giảm số trạng thái.
3. Nắm **định lý Bouton**: trong Nim, người đi trước thắng ⇔ `p₁ ⊕ p₂ ⊕ … ⊕ pₙ ≠ 0`, và biết cách **tìm nước thắng**.
4. Làm quen **số Grundy (Sprague–Grundy)** — công cụ tổng quát cho tổng các trò chơi độc lập.
5. Kiểm chứng công thức XOR bằng **vét cạn toàn bộ** miền ràng buộc (`n ≤ 7`, `pᵢ ≤ 7`).

## 2. Đề bài & ví dụ

Alice và Bob lần lượt chơi, **Alice đi trước**. Có `n` đống đá. Đến lượt mình, người chơi phải bốc **một số dương bất kỳ** viên đá từ **một** đống còn đá (tùy chọn đống). Người **không đi được** (mọi đống đã hết) **thua**; người kia thắng.

Cho mảng `piles` (`piles[i]` là số đá của đống `i`), trả về `true` nếu Alice thắng, `false` nếu Bob thắng. Cả hai chơi **tối ưu**.

**Ví dụ 1:** `piles = [1]` → `true` (Alice bốc viên duy nhất, Bob hết nước).

**Ví dụ 2:** `piles = [1,1]` → `false` (Alice bốc một đống, Bob bốc đống còn lại, Alice hết nước).

**Ví dụ 3:** `piles = [1,2,3]` → `false`.

**Ràng buộc:** `1 ≤ n ≤ 7`, `1 ≤ piles[i] ≤ 7`.

**Follow-up:** có lời giải thời gian tuyến tính không? (Đề nói có thể vượt phạm vi phỏng vấn nhưng thú vị.)

In [1]:
import random
import time
from functools import lru_cache, reduce
from itertools import combinations_with_replacement, product
from operator import xor
from typing import List, Tuple

examples = [([1], True), ([1, 1], False), ([1, 2, 3], False)]
for piles, want in examples:
    print(piles, "-> Alice thắng?" , want)

[1] -> Alice thắng? True
[1, 1] -> Alice thắng? False
[1, 2, 3] -> Alice thắng? False


## 3. Kiến thức nền

### 3.1 Trạng thái thắng / thua

Định nghĩa đệ quy cho trò chơi "người không đi được thì thua":
- Trạng thái **không có nước đi** là **thua** (cho người sắp đi).
- Trạng thái là **thắng** nếu **có ít nhất một** nước đi tới trạng thái **thua** của đối thủ.
- Ngược lại (mọi nước đi đều tới trạng thái thắng của đối thủ) là **thua**.

In [2]:
# Ví dụ nhỏ: một đống k viên. k = 0 thua; k >= 1 thắng (bốc hết).
def one_pile_win(k):
    return any(not one_pile_win(k - t) for t in range(1, k + 1))

print([one_pile_win(k) for k in range(6)])

[False, True, True, True, True, True]


### 3.2 Tuple làm khóa ghi nhớ, và chuẩn hóa bằng `sorted`

`lru_cache` cần đối số **băm được** — list không băm được, tuple thì được. Thứ tự các đống **không** ảnh hưởng kết quả, nên `tuple(sorted(piles))` gộp các trạng thái hoán vị lại một.

In [3]:
s1, s2 = (3, 1, 2), (1, 2, 3)
print(tuple(sorted(s1)) == tuple(sorted(s2)))
try:
    {[1, 2]: True}
except TypeError as e:
    print("list không làm khóa được:", e)

True
list không làm khóa được: unhashable type: 'list'


### 3.3 Phép XOR `^`

XOR từng bit: `1 ^ 1 = 0`, `1 ^ 0 = 1`. Tính chất: giao hoán, kết hợp, `a ^ a = 0`, `a ^ 0 = a`. `reduce(xor, piles, 0)` tính XOR cả mảng.

In [4]:
print(1 ^ 2 ^ 3, bin(1), bin(2), bin(3))
print(reduce(xor, [3, 4, 5], 0), "=", bin(3 ^ 4 ^ 5))

0 0b1 0b10 0b11
2 = 0b10


### 3.4 Bit cao nhất và `bit_length`

`x.bit_length() - 1` là vị trí bit 1 cao nhất của `x > 0`. Dùng để tìm đống "có thể sửa" khi muốn đưa XOR về 0: đống có bit đó bằng 1.

In [5]:
x = 0b0110
hb = x.bit_length() - 1
print("bit cao nhất của", bin(x), "là vị trí", hb, "| các đống có bit đó:", [p for p in [3, 4, 5] if p >> hb & 1])

bit cao nhất của 0b110 là vị trí 2 | các đống có bit đó: [4, 5]


### 3.5 Mã hóa trạng thái bằng cơ số 8

Mỗi đống ≤ 7 vừa đúng 3 bit ⇒ trạng thái `n` đống là số nguyên `Σ pᵢ · 8ⁱ` < `8⁷ = 2097152`. Cho phép DP **mảng** thay vì `dict`.

In [6]:
def encode(piles):
    code = 0
    for i, p in enumerate(piles):
        code |= p << (3 * i)
    return code

def decode(code, n):
    return [(code >> (3 * i)) & 7 for i in range(n)]

c = encode([1, 2, 3])
print(c, decode(c, 3))

209 [1, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

Thử tay các trạng thái nhỏ (đã sắp xếp), ký hiệu W = người sắp đi thắng, L = thua:

| trạng thái | các nước đi tới | kết luận |
|---|---|---|
| `()` / `(0,…)` | không có | **L** |
| `(1)` | `()` là L | **W** |
| `(1,1)` | chỉ tới `(1)` là W | **L** |
| `(2)` | tới `(1)` W hoặc `()` L | **W** (bốc hết) |
| `(1,2)` | tới `(2)` W, `(1)` W, `(1,1)` **L** | **W** (bốc 1 từ đống 2) |
| `(2,2)` | tới `(1,2)` W, `(2)` W | **L** |
| `(1,2,3)` | mọi nước đều tới W (kiểm bằng code) | **L** |

Nhận xét: `(1,1)`, `(2,2)`, `(1,2,3)` thua — và `1⊕1 = 0`, `2⊕2 = 0`, `1⊕2⊕3 = 0`! Các trạng thái thắng có XOR ≠ 0. Đây chính là định lý Bouton.

In [7]:
for st in [(1,), (1, 1), (2,), (1, 2), (2, 2), (1, 2, 3), (3, 4, 5)]:
    print(st, "XOR =", reduce(xor, st, 0))

(1,) XOR = 1
(1, 1) XOR = 0
(2,) XOR = 2
(1, 2) XOR = 3
(2, 2) XOR = 0
(1, 2, 3) XOR = 0
(3, 4, 5) XOR = 2


## 5. Cách 1 — Brute Force / baseline: minimax đệ quy không ghi nhớ

**Ý tưởng:** đúng định nghĩa ở 3.1 — thử mọi nước (chọn đống `i`, bốc `t = 1..pᵢ`), nếu có nước dẫn tới trạng thái mà đối thủ thua thì ta thắng.

In [8]:
def win_naive(piles):
    piles = list(piles)
    for i, p in enumerate(piles):
        for t in range(1, p + 1):
            piles[i] -= t
            opp = win_naive(piles)
            piles[i] += t                 # trả lại trạng thái (quay lui)
            if not opp:
                return True
    return False

calls = [0]
def win_naive_counted(piles):
    calls[0] += 1
    piles = list(piles)
    for i, p in enumerate(piles):
        for t in range(1, p + 1):
            piles[i] -= t
            opp = win_naive_counted(piles)
            piles[i] += t
            if not opp:
                return True
    return False

for piles, want in examples:
    print(piles, win_naive(piles), "| kỳ vọng", want)
for st in ([2, 2, 2, 2], [3, 3, 3, 3], [4, 4, 3, 3]):
    calls[0] = 0; t = time.perf_counter(); r = win_naive_counted(st)
    print(st, r, f"| {calls[0]} lời gọi, {time.perf_counter() - t:.2f}s")

[1] True | kỳ vọng True
[1, 1] False | kỳ vọng False
[1, 2, 3] False | kỳ vọng False
[2, 2, 2, 2] False | 1403 lời gọi, 0.00s
[3, 3, 3, 3] False | 60615 lời gọi, 0.03s


[4, 4, 3, 3] False | 887073 lời gọi, 0.36s


**Độ phức tạp:** cây trò chơi phình theo cấp số nhân — cùng một trạng thái bị tính lại vô số lần. Để ý các trạng thái **thua** (XOR = 0) là tệ nhất: phải duyệt **hết** mọi nước mới kết luận được. `[3,3,3,3]` cần ~6·10⁴ lời gọi, `[4,4,3,3]` ~9·10⁵, `[4,4,4,4]` đã ~1.8·10⁷ (vài giây); `[7]*6` (tổng 42 viên, cũng là trạng thái thua) là bất khả.

**Nút thắt:** **trùng lặp trạng thái** ⇒ ghi nhớ (memoization).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — số trạng thái nhỏ:** mỗi đống `0..7`, tối đa 7 đống ⇒ `≤ 8⁷ ≈ 2·10⁶` trạng thái có thứ tự, và chỉ `C(7+7, 7) = 3432` trạng thái **không thứ tự** (multiset). Ghi nhớ trên trạng thái đã sắp xếp là đủ nhanh.

**Insight 2 — định lý Bouton (1901):** gọi `X = p₁ ⊕ … ⊕ pₙ`.
- *Nếu* `X = 0` *thì* **mọi** nước đi làm `X ≠ 0` (đổi một đống từ `p` thành `p' ≠ p` thì `X` đổi thành `p ⊕ p' ≠ 0`).
- *Nếu* `X ≠ 0` *thì* **tồn tại** nước đi làm `X = 0`: lấy đống `p` có bit cao nhất của `X` bằng 1; khi đó `p ⊕ X < p`, bốc `p − (p ⊕ X)` viên để đống còn `p ⊕ X`.
- Trạng thái cuối (mọi đống 0) có `X = 0` và là thua.

⇒ trạng thái `X = 0` là **L**, `X ≠ 0` là **W** (quy nạp theo tổng số đá).

**Insight 3 — Sprague–Grundy:** mỗi đống `p` là một trò chơi có số Grundy `g(p) = p`; tổng các trò chơi độc lập có Grundy = XOR các Grundy. Trạng thái thua ⇔ Grundy = 0.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Minimax có ghi nhớ trên trạng thái chuẩn hóa

In [9]:
@lru_cache(maxsize=None)
def win_memo(state: Tuple[int, ...]) -> bool:
    # state: tuple đã sắp xếp, đã bỏ các đống 0
    for i, p in enumerate(state):
        if i > 0 and state[i - 1] == p:
            continue                                  # đống trùng giá trị: nước đi tương đương
        for t in range(1, p + 1):
            nxt = list(state)
            nxt[i] = p - t
            nxt = tuple(sorted(x for x in nxt if x))
            if not win_memo(nxt):
                return True
    return False

class SolutionMemo:
    def nimGame(self, piles: List[int]) -> bool:
        return win_memo(tuple(sorted(x for x in piles if x)))

for piles, want in examples:
    print(SolutionMemo().nimGame(piles), want)
t = time.perf_counter(); r = SolutionMemo().nimGame([7] * 7)
print("[7]*7:", r, f"| {time.perf_counter() - t:.3f}s | số trạng thái đã nhớ: {win_memo.cache_info().currsize}")

True True
False False
False False
[7]*7: True | 0.006s | số trạng thái đã nhớ: 1723


**Độ phức tạp:** số trạng thái `S ≤ 3432`, mỗi trạng thái thử `≤ Σpᵢ ≤ 49` nước, mỗi nước sắp xếp `O(n log n)` ⇒ cỡ `3432 · 49 · 7` — rất nhanh. Bộ nhớ `O(S)`.

### 7.2 Cách 3 — DP từ dưới lên trên mã cơ số 8

Tính `win[code]` cho mọi trạng thái **có thứ tự** của đúng `n` đống, theo thứ tự `code` tăng — mọi nước đi làm một đống giảm ⇒ `code` giảm ⇒ đã tính trước.

In [10]:
def win_table(n):
    size = 8 ** n
    win = bytearray(size)                       # 0 = thua, 1 = thắng
    for code in range(1, size):
        for i in range(n):
            p = (code >> (3 * i)) & 7
            base = code - (p << (3 * i))         # đống i về 0
            for q in range(p):                   # đống i còn q < p viên
                if not win[base | (q << (3 * i))]:
                    win[code] = 1
                    break
            if win[code]:
                break
    return win

class SolutionTable:
    _cache = {}
    def nimGame(self, piles: List[int]) -> bool:
        n = len(piles)
        if n not in self._cache:
            self._cache[n] = win_table(n)
        return bool(self._cache[n][encode(piles)])

t = time.perf_counter(); tbl4 = win_table(4)
print(f"bảng n=4: {len(tbl4)} trạng thái trong {time.perf_counter() - t:.2f}s")
for piles, want in examples:
    print(SolutionTable().nimGame(piles), want)

bảng n=4: 4096 trạng thái trong 0.00s
True True
False False
False False


**Độ phức tạp:** `8ⁿ` trạng thái × `≤ 7n` nước ⇒ `n = 7` là `2·10⁶ · 49 ≈ 10⁸` phép Python — quá chậm cho notebook (ta chỉ dựng tới `n = 4`). Bài học: chuẩn hóa trạng thái (Cách 2) quan trọng hơn "mảng nhanh hơn dict".

### 7.3 Cách 4 — XOR (định lý Bouton), `O(n)` — lời giải nộp

In [11]:
class SolutionXor:
    def nimGame(self, piles: List[int]) -> bool:
        x = 0
        for p in piles:
            x ^= p
        return x != 0

for piles, want in examples:
    print(SolutionXor().nimGame(piles), want)

True True
False False
False False


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ — đúng câu trả lời cho follow-up.

### 7.4 Bổ sung — tìm **nước thắng** (không chỉ trả lời thắng/thua)

Khi `X ≠ 0`, chọn đống có bit cao nhất của `X`, đưa nó về `p ⊕ X`.

In [12]:
def winning_move_demo(piles):
    X = reduce(xor, piles, 0)
    if X == 0:
        return None
    hb = X.bit_length() - 1
    for i, p in enumerate(piles):
        if p >> hb & 1:
            return i, p - (p ^ X)          # (đống, số viên cần bốc)

for st in ([3, 4, 5], [1, 2], [7, 1], [1, 2, 3]):
    mv = winning_move_demo(st)
    if mv:
        i, take = mv; after = st[:]; after[i] -= take
        print(st, "-> bốc", take, "viên ở đống", i, "->", after, "XOR =", reduce(xor, after, 0))
    else:
        print(st, "-> XOR = 0, không có nước thắng")

[3, 4, 5] -> bốc 2 viên ở đống 0 -> [1, 4, 5] XOR = 0
[1, 2] -> bốc 1 viên ở đống 1 -> [1, 1] XOR = 0
[7, 1] -> bốc 6 viên ở đống 0 -> [1, 1] XOR = 0
[1, 2, 3] -> XOR = 0, không có nước thắng


### 7.5 Góc nhìn Sprague–Grundy: tính số Grundy bằng `mex`

**Số Grundy** của một trạng thái: `g(s) = mex{ g(s') : s' là trạng thái con }`, với `mex` (minimum excludant) là số tự nhiên nhỏ nhất **không** có trong tập. Trạng thái thua ⇔ `g = 0`. **Định lý Sprague–Grundy:** tổng của các trò chơi độc lập (mỗi lượt chọn **một** thành phần để đi) có Grundy bằng **XOR** các Grundy thành phần.

Với Nim, một đống `p` có các con `0, 1, …, p−1` ⇒ `g(p) = mex{g(0), …, g(p−1)}` = `p` (quy nạp). Ta kiểm bằng code, đồng thời kiểm "Grundy của cả bàn = XOR" bằng cách tính trực tiếp Grundy trên trạng thái nhiều đống.

In [13]:
def mex(values):
    g = 0
    while g in values:
        g += 1
    return g

@lru_cache(maxsize=None)
def grundy_state(state):
    children = set()
    for i, p in enumerate(state):
        for t in range(1, p + 1):
            children.add(grundy_state(tuple(sorted(x for x in state[:i] + (p - t,) + state[i + 1:] if x))))
    return mex(children)

print("Grundy một đống p = 0..7:", [grundy_state((p,) if p else ()) for p in range(8)])
for st in [(1, 2), (1, 2, 3), (3, 4, 5), (2, 5, 7)]:
    print(st, "Grundy trực tiếp =", grundy_state(st), "| XOR =", reduce(xor, st, 0))
assert all(grundy_state(tuple(sorted(st))) == reduce(xor, st, 0)
           for st in combinations_with_replacement(range(1, 6), 3))
print("Grundy(bàn) = XOR(Grundy từng đống) trên mọi bộ 3 đống ≤ 5")

Grundy một đống p = 0..7: [0, 1, 2, 3, 4, 5, 6, 7]
(1, 2) Grundy trực tiếp = 3 | XOR = 3
(1, 2, 3) Grundy trực tiếp = 0 | XOR = 0
(3, 4, 5) Grundy trực tiếp = 2 | XOR = 2
(2, 5, 7) Grundy trực tiếp = 0 | XOR = 0
Grundy(bàn) = XOR(Grundy từng đống) trên mọi bộ 3 đống ≤ 5


Ý nghĩa thực tế: nếu đề đổi luật cho **từng đống** (ví dụ chỉ bốc 1..k viên, hay chỉ bốc số chẵn), ta chỉ cần tính Grundy của **một đống** theo luật mới rồi XOR — không cần minimax trên cả bàn.

## 8. TRACE từng bước

### 8.1 Trace minimax có ghi nhớ trên `[1, 2]`

| trạng thái | nước thử | trạng thái con | con thắng? | kết luận |
|---|---|---|---|---|
| (1,2) | bốc 1 từ đống 1 | (2) | W | thử tiếp |
| (1,2) | bốc 1 từ đống 2 | (1,1) | **L** | **W** — dừng |

### 8.2 Trace XOR + nước thắng trên `[3, 4, 5]`

`3 = 011`, `4 = 100`, `5 = 101` ⇒ `X = 010` (= 2). Bit cao nhất của `X` là bit 1; đống `3 = 011` có bit 1 ⇒ đổi `3 → 3 ⊕ 2 = 1` (bốc 2 viên) ⇒ `[1,4,5]`, `1 ⊕ 4 ⊕ 5 = 0`.

In [14]:
def trace_minimax(state, depth=0, memo=None):
    memo = {} if memo is None else memo
    pad = "  " * depth
    if state in memo:
        print(f"{pad}{state}: (đã nhớ) {'W' if memo[state] else 'L'}"); return memo[state]
    for i, p in enumerate(state):
        for t in range(1, p + 1):
            nxt = tuple(sorted(x for x in state[:i] + (p - t,) + state[i + 1:] if x))
            print(f"{pad}{state}: bốc {t} từ đống {p} -> {nxt}")
            if not trace_minimax(nxt, depth + 1, memo):
                print(f"{pad}=> {state} là W (đối thủ rơi vào L)"); memo[state] = True; return True
    print(f"{pad}=> {state} là L (mọi nước đều tới W)"); memo[state] = False; return False

trace_minimax((1, 2))

(1, 2): bốc 1 từ đống 1 -> (2,)
  (2,): bốc 1 từ đống 2 -> (1,)
    (1,): bốc 1 từ đống 1 -> ()
      => () là L (mọi nước đều tới W)
    => (1,) là W (đối thủ rơi vào L)
  (2,): bốc 2 từ đống 2 -> ()
    (): (đã nhớ) L
  => (2,) là W (đối thủ rơi vào L)
(1, 2): bốc 1 từ đống 2 -> (1, 1)
  (1, 1): bốc 1 từ đống 1 -> (1,)
    (1,): (đã nhớ) W
  (1, 1): bốc 1 từ đống 1 -> (1,)
    (1,): (đã nhớ) W
  => (1, 1) là L (mọi nước đều tới W)
=> (1, 2) là W (đối thủ rơi vào L)


True

In [15]:
def trace_xor(piles):
    X = 0
    for p in piles:
        print(f"  {X:03b} ^ {p:03b} = {X ^ p:03b}")
        X ^= p
    print("XOR =", X, "->", "Alice thắng" if X else "Bob thắng")
    if X:
        i, take = winning_move_demo(piles)
        print(f"nước thắng: đống {i} ({piles[i]} = {piles[i]:03b}) -> {piles[i] ^ X} (bốc {take})")
    return X != 0

trace_xor([3, 4, 5]); print(); trace_xor([1, 2, 3])

  000 ^ 011 = 011
  011 ^ 100 = 111
  111 ^ 101 = 010
XOR = 2 -> Alice thắng
nước thắng: đống 0 (3 = 011) -> 1 (bốc 2)

  000 ^ 001 = 001
  001 ^ 010 = 011
  011 ^ 011 = 000
XOR = 0 -> Bob thắng


False

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def nimGame(self, piles: List[int]) -> bool:
        x = 0
        for p in piles:
            x ^= p               # định lý Bouton: thua <=> XOR = 0
        return x != 0

for piles, want in examples:
    assert Solution().nimGame(piles) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Chứng minh định lý Bouton bằng quy nạp theo tổng số đá `T`.

**(a) Từ `X = 0`, mọi nước đi đều tới `X' ≠ 0`.** Đổi đống `p` thành `p' < p`: `X' = X ⊕ p ⊕ p' = p ⊕ p' ≠ 0` vì `p ≠ p'`.

**(b) Từ `X ≠ 0`, có nước đi tới `X' = 0`.** Gọi `k` là bit cao nhất của `X`. Có ít nhất một đống `p` có bit `k` bằng 1 (vì bit `k` của `X` là XOR các bit `k`). Đặt `p' = p ⊕ X`: bit `k` của `p'` là 0, các bit cao hơn giữ nguyên ⇒ `p' < p`, là nước đi hợp lệ (bốc `p − p' > 0` viên), và `X' = X ⊕ p ⊕ p' = 0`.

**(c) Quy nạp:** trạng thái `T = 0` có `X = 0` và là **thua** (không đi được). Giả sử khẳng định đúng cho mọi tổng `< T`. Nếu `X ≠ 0`: theo (b) ta đi tới `X' = 0` — trạng thái thua của đối thủ ⇒ **thắng**. Nếu `X = 0`: theo (a) mọi nước đều tới `X' ≠ 0` — trạng thái thắng của đối thủ ⇒ **thua**.

Mục 12 còn kiểm chứng trên **mọi** multiset hợp lệ của ràng buộc (`n ≤ 7`, mỗi đống `1..7`).

## 11. Phân tích Time / Space

Gọi `n ≤ 7` = số đống, `M = 7` = số đá tối đa mỗi đống.

| Cách | Time | Space |
|---|---|---|
| 1. Minimax không nhớ | hàm mũ theo `Σ pᵢ` | `O(Σ pᵢ)` stack |
| 2. Minimax + nhớ (chuẩn hóa) | `O(S · Σpᵢ · n log n)`, `S = C(n+M, n) ≤ 3432` | `O(S)` |
| 3. DP bảng cơ số 8 | `O((M+1)ⁿ · n · M)` | `O((M+1)ⁿ)` |
| 4. XOR | `O(n)` | `O(1)` |

In [17]:
rng = random.Random(1908)
tests = [[rng.randint(1, 7) for _ in range(rng.randint(1, 7))] for _ in range(2000)]
for name, sol in [("Cách 2 nhớ", SolutionMemo()), ("Cách 4 XOR / nộp", Solution())]:
    t = time.perf_counter()
    for p in tests:
        sol.nimGame(p)
    print(f"{name}: 2000 truy vấn trong {1000*(time.perf_counter()-t):.1f} ms")

Cách 2 nhớ: 2000 truy vấn trong 6.6 ms
Cách 4 XOR / nộp: 2000 truy vấn trong 0.2 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
sols = [Solution(), SolutionXor(), SolutionMemo()]
edge = examples + [
    ([7], True), ([7, 7], False), ([1, 2], True), ([2, 2], False),
    ([3, 4, 5], True), ([1, 1, 1], True), ([1, 1, 1, 1], False),
    ([7] * 7, True), ([7] * 6, False), ([1, 2, 4, 7], False),
]
for piles, want in edge:
    assert win_naive(piles) == want if sum(piles) <= 9 else True
    for s in sols:
        assert s.nimGame(piles) == want, (piles, s)
    assert SolutionTable().nimGame(piles) == want if len(piles) <= 3 else True
    print("OK", piles, "->", want)

OK [1] -> True
OK [1, 1] -> False
OK [1, 2, 3] -> False
OK [7] -> True
OK [7, 7] -> False
OK [1, 2] -> True
OK [2, 2] -> False
OK [3, 4, 5] -> True
OK [1, 1, 1] -> True
OK [1, 1, 1, 1] -> False
OK [7, 7, 7, 7, 7, 7, 7] -> True
OK [7, 7, 7, 7, 7, 7] -> False
OK [1, 2, 4, 7] -> False


(Các kỳ vọng tự dựng được kiểm bằng minimax có ghi nhớ — chuẩn độc lập với công thức XOR — và bằng minimax không nhớ khi tổng nhỏ.)

### 12.2 Kiểm chứng **toàn bộ** miền ràng buộc

Mọi multiset `n ≤ 7` đống, mỗi đống `1..7`: so XOR với minimax có ghi nhớ.

In [19]:
from itertools import combinations_with_replacement
total = 0
for n in range(1, 8):
    for piles in combinations_with_replacement(range(1, 8), n):
        assert Solution().nimGame(list(piles)) == win_memo(tuple(sorted(piles)))
        total += 1
print("đã kiểm", total, "multiset — XOR khớp minimax")
tbl3 = win_table(3)
for piles in product(range(8), repeat=3):
    assert bool(tbl3[encode(list(piles))]) == (piles[0] ^ piles[1] ^ piles[2] != 0)
print("bảng DP n=3 (512 trạng thái có thứ tự) khớp XOR")

đã kiểm 3431 multiset — XOR khớp minimax
bảng DP n=3 (512 trạng thái có thứ tự) khớp XOR


### 12.3 Nước thắng luôn hợp lệ và đưa XOR về 0

In [20]:
rng = random.Random(7)
for _ in range(2000):
    piles = [rng.randint(1, 7) for _ in range(rng.randint(1, 7))]
    mv = winning_move_demo(piles)
    if Solution().nimGame(piles):
        i, take = mv
        assert 1 <= take <= piles[i]
        after = piles[:]; after[i] -= take
        assert reduce(xor, after, 0) == 0 and not win_memo(tuple(sorted(x for x in after if x)))
    else:
        assert mv is None
print("nước thắng đúng trên 2000 trạng thái ngẫu nhiên")

nước thắng đúng trên 2000 trạng thái ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đoán theo **tính chẵn lẻ của tổng** số đá

In [21]:
def wrong_parity(piles):
    return sum(piles) % 2 == 1          # SAI

for p in ([2], [1, 2, 3], [3, 4, 5], [1, 1]):
    print(p, "sai:", wrong_parity(p), "| đúng:", Solution().nimGame(p))

[2] sai: False | đúng: True
[1, 2, 3] sai: False | đúng: False
[3, 4, 5] sai: False | đúng: True
[1, 1] sai: False | đúng: False


**Vì sao sai:** đúng nếu mỗi lượt chỉ được bốc **1** viên, nhưng ở đây bốc bao nhiêu cũng được. `[2]`: tổng chẵn nhưng Alice bốc hết và thắng. **Sửa:** XOR, không phải tổng.

### Lỗi 2 — Đảo logic: "thắng nếu có con **thắng**"

In [22]:
def wrong_logic(piles):
    piles = list(piles)
    for i, p in enumerate(piles):
        for t in range(1, p + 1):
            piles[i] -= t
            child = wrong_logic(piles)
            piles[i] += t
            if child:                     # SAI: phải là "not child"
                return True
    return False

for p, want in examples:
    print(p, "sai:", wrong_logic(p), "| đúng:", want)

[1] sai: False | đúng: True
[1, 1] sai: False | đúng: False
[1, 2, 3] sai: False | đúng: False


**Vì sao sai:** trạng thái con là lượt của **đối thủ**; ta thắng khi đưa đối thủ vào trạng thái **thua** (`not child`). Phiên bản sai luôn trả `False` (lá trả `False`, nên mọi nút đều `False`).

### Lỗi 3 — Quay lui quên trả lại trạng thái

In [23]:
def wrong_no_restore(piles):
    piles = list(piles)
    for i, p in enumerate(piles):
        for t in range(1, p + 1):
            piles[i] -= t                 # SAI: không cộng lại t sau khi thử
            if not wrong_no_restore(piles):
                return True
    return False

for p in ([2, 2], [3, 3], [1, 2, 3]):
    print(p, "sai:", wrong_no_restore(p), "| đúng:", Solution().nimGame(p))

[2, 2] sai: True | đúng: False
[3, 3] sai: True | đúng: False
[1, 2, 3] sai: True | đúng: False


**Vì sao sai:** sau nước thử đầu tiên, `piles[i]` đã bị trừ; nước thử thứ hai trừ **thêm** — đống có thể âm và các nước thử không còn xuất phát từ cùng một trạng thái. **Sửa:** `piles[i] += t` sau mỗi lần thử (hoặc tạo bản sao mới).

### Lỗi 4 — Dùng list làm khóa ghi nhớ

In [24]:
@lru_cache(maxsize=None)
def memo_bad(state):
    return any(not memo_bad(state[:i] + [p - t] + state[i + 1:]) for i, p in enumerate(state) for t in range(1, p + 1))

try:
    memo_bad([1, 2])
except TypeError as e:
    print("TypeError:", e, "| sửa: dùng tuple ->", SolutionMemo().nimGame([1, 2]))

TypeError: unhashable type: 'list' | sửa: dùng tuple -> True


**Vì sao sai:** `lru_cache` băm đối số; list khả biến nên không băm được. **Sửa:** `tuple(...)` (và nên `sorted` để gộp hoán vị).

### Lỗi 5 — Nghĩ rằng "số đống lẻ thì thắng" hoặc "đống lớn nhất quyết định"

In [25]:
def wrong_count(piles):
    return len(piles) % 2 == 1            # SAI

for p in ([1, 2, 3], [2, 2], [1, 2]):
    print(p, "sai:", wrong_count(p), "| đúng:", Solution().nimGame(p))

[1, 2, 3] sai: True | đúng: False
[2, 2] sai: False | đúng: False
[1, 2] sai: False | đúng: True


**Vì sao sai:** `[1,2,3]` có 3 đống (lẻ) nhưng thua; `[1,2]` có 2 đống nhưng thắng. Chỉ đúng trong trường hợp đặc biệt mọi đống bằng 1.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Trò chơi hai người, chơi tối ưu, "không đi được thì thua"** ⇒ trạng thái W/L; ràng buộc nhỏ ⇒ minimax + ghi nhớ.
- **Nhiều "đống" độc lập, mỗi lượt chọn một đống** ⇒ Sprague–Grundy: XOR các số Grundy của từng đống.
- Nim chuẩn (bốc tùy ý) ⇒ Grundy của đống = số đá ⇒ XOR các đống.
- Ghi nhớ: tuple + chuẩn hóa (sắp xếp) để giảm trạng thái.
- Luôn kiểm công thức "đoán" bằng vét cạn trên miền nhỏ.

## 15. Biến thể của bài

1. **Mỗi lượt bốc 1..k viên** (subtraction game) ⇒ Grundy của đống `p` là `p mod (k+1)` (Bài tập 4).
2. **Misère Nim** (người bốc viên cuối **thua**) ⇒ nếu mọi đống ≤ 1: thắng ⇔ số đống 1 là **chẵn**; ngược lại như Nim thường (Bài tập 5).
3. **LeetCode 292 — Nim Game:** một đống, bốc 1–3 ⇒ thua ⇔ `n % 4 == 0`.
4. **LeetCode 877/1140 Stone Game** — không phải Nim (tính điểm) ⇒ DP minimax trên đoạn.
5. **Staircase Nim, Moore's Nimₖ** — biến thể dùng XOR trên các vị trí lẻ / XOR cơ số `k+1`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax không nhớ | định nghĩa W/L thuần | mũ | `O(T)` | chỉ để hiểu |
| 2. Minimax + nhớ | trạng thái đã sắp xếp | `~10⁵` phép | `O(3432)` | trò chơi lạ, chưa có công thức |
| 3. DP bảng | mã cơ số 8, từ dưới lên | `O(8ⁿ·7n)` | `O(8ⁿ)` | khi trạng thái có thứ tự thật sự khác nhau |
| 4. XOR | Bouton / Sprague–Grundy | `O(n)` | `O(1)` | **nộp bài**, follow-up |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[3,3]`, `[1,2]`, `[5,5,1]`?

**Bài 2 (tính tay):** Với `[3,4,5]`, tính XOR theo nhị phân và tìm **tất cả** các nước thắng.

**Bài 3 (điền TODO):** Viết `all_winning_moves(piles)` trả về list mọi `(i, số_viên_bốc)` đưa XOR về 0.

**Bài 4 (tự viết lại):** Trò chơi mỗi lượt bốc **1..k** viên từ một đống. Viết `grundy_subtraction(p, k)` bằng định nghĩa (mex) và `win_subtraction(piles, k)`; kiểm tra công thức `p mod (k+1)`.

**Bài 5 (mini challenge — Misère Nim):** Người bốc viên **cuối cùng thua**. Viết `misere_win(piles)` theo công thức và đối chiếu với minimax.

In [26]:
def all_winning_moves(piles):
    # TODO
    pass

def grundy_subtraction(p, k):
    # TODO: mex của {g(p - t) | 1 <= t <= min(k, p)}
    pass

def misere_win(piles):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Tính XOR | `3⊕3 = 0` | `5⊕5⊕1 = 1` ⇒ thắng |
| 2 | `X = 2` | Đống có bit 1: chỉ `3 (011)` | Chỉ một nước: bốc 2 từ đống 3 |
| 3 | Duyệt mọi đống `p` | Nếu `p ⊕ X < p` thì là nước thắng | Bốc `p − (p ⊕ X)` |
| 4 | `mex(S)` = số tự nhiên nhỏ nhất không thuộc `S` | Ghi nhớ `g` theo `p` | XOR các `g(p)` ≠ 0 ⇒ thắng |
| 5 | Trường hợp mọi đống ≤ 1 | Có đống ≥ 2 ⇒ giống Nim thường | Minimax với lá là **thắng** (đối thủ vừa bốc viên cuối) |

## 19. Đáp án tham khảo

In [27]:
r1 = [Solution().nimGame(p) for p in ([3, 3], [1, 2], [5, 5, 1])]
print("Bài 1:", r1); assert r1 == [False, True, True]
print("Bài 2:"); trace_xor([3, 4, 5])

Bài 1: [False, True, True]
Bài 2:
  000 ^ 011 = 011
  011 ^ 100 = 111
  111 ^ 101 = 010
XOR = 2 -> Alice thắng
nước thắng: đống 0 (3 = 011) -> 1 (bốc 2)


True

In [28]:
def all_winning_moves(piles):
    X = reduce(xor, piles, 0)
    return [(i, p - (p ^ X)) for i, p in enumerate(piles) if X and (p ^ X) < p]

def all_winning_moves_brute(piles):
    out = []
    for i, p in enumerate(piles):
        for t in range(1, p + 1):
            after = tuple(sorted(x for x in piles[:i] + [p - t] + piles[i + 1:] if x))
            if not win_memo(after):
                out.append((i, t))
    return out

@lru_cache(maxsize=None)
def grundy_subtraction(p, k):
    s = {grundy_subtraction(p - t, k) for t in range(1, min(k, p) + 1)}
    g = 0
    while g in s:
        g += 1
    return g

def win_subtraction(piles, k):
    return reduce(xor, (grundy_subtraction(p, k) for p in piles), 0) != 0

assert all_winning_moves([3, 4, 5]) == [(0, 2)]
rng = random.Random(3)
for _ in range(500):
    piles = [rng.randint(1, 7) for _ in range(rng.randint(1, 5))]
    assert all_winning_moves(piles) == all_winning_moves_brute(piles)
for k in range(1, 6):
    assert all(grundy_subtraction(p, k) == p % (k + 1) for p in range(40))

@lru_cache(maxsize=None)
def win_sub_brute(state, k):
    return any(not win_sub_brute(tuple(sorted(x for x in state[:i] + (p - t,) + state[i + 1:] if x)), k)
               for i, p in enumerate(state) for t in range(1, min(k, p) + 1))
for _ in range(300):
    piles = [rng.randint(1, 7) for _ in range(rng.randint(1, 4))]; k = rng.randint(1, 4)
    assert win_subtraction(piles, k) == win_sub_brute(tuple(sorted(piles)), k)
print("Bài 3, 4: OK", all_winning_moves([1, 2, 4]))

Bài 3, 4: OK [(2, 1)]


In [29]:
def misere_win(piles):
    if all(p <= 1 for p in piles):
        return sum(piles) % 2 == 0          # mọi đống 0/1: thắng <=> số đống 1 chẵn
    return reduce(xor, piles, 0) != 0

@lru_cache(maxsize=None)
def misere_brute(state):
    if not state:
        return True                          # đối thủ vừa bốc viên cuối -> đối thủ thua -> ta thắng
    return any(not misere_brute(tuple(sorted(x for x in state[:i] + (p - t,) + state[i + 1:] if x)))
               for i, p in enumerate(state) for t in range(1, p + 1))

for n in range(1, 6):
    for piles in combinations_with_replacement(range(1, 7), n):
        assert misere_win(list(piles)) == misere_brute(tuple(piles)), piles
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", misere_win([1]), misere_win([1, 1]), misere_win([2, 2]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK False True False


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Nim chuẩn: người không đi được thua
x = 0
for p in piles: x ^= p
return x != 0

# Khung minimax khi chưa biết công thức
@lru_cache(None)
def win(state):                       # state: tuple đã chuẩn hóa
    return any(not win(nxt) for nxt in moves(state))
```

**Checklist:**
- [ ] Hiểu đúng luật: bốc **bao nhiêu cũng được** từ **một** đống (không phải 1..k)?
- [ ] Người **không đi được** thua (Nim thường, không phải misère)?
- [ ] Minimax: thắng nếu **tồn tại** nước đưa đối thủ vào **thua** (`not child`)?
- [ ] Ghi nhớ bằng tuple (đã sắp xếp), quay lui trả lại trạng thái?
- [ ] Đã kiểm XOR với vét cạn trên toàn miền nhỏ.